In [65]:
import torch
from torch import nn
from typing import Optional, Tuple


In [66]:
class KVCacheManager:
    def __init__(self, n_layers, n_heads, max_seq_len, dim_head, device, batch_size):
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        self.dim_head = dim_head
        self.device = device
        self.batch_size = batch_size

        # preallocate KV cache for all layers, now with batch_size
        self.K = [torch.zeros(batch_size, n_heads, max_seq_len, dim_head, device=device) for _ in range(n_layers)]
        self.V = [torch.zeros(batch_size, n_heads, max_seq_len, dim_head, device=device) for _ in range(n_layers)]
        self.curr_len = 0  # tokens stored

    def update(self, layer_id, K_new, V_new):
        # K_new, V_new are expected to be (B, n_heads, T_new, dim_head)
        B, num_heads, T_new, dim_head = K_new.shape
        assert B == self.batch_size
        assert num_heads == self.n_heads
        assert dim_head == self.dim_head

        # Ensure there's space in the cache
        if self.curr_len + T_new > self.max_seq_len:
            raise ValueError("KV cache out of capacity!")

        # Update the cache for the current layer and for the new tokens
        self.K[layer_id][:, :, self.curr_len : self.curr_len + T_new, :] = K_new
        self.V[layer_id][:, :, self.curr_len : self.curr_len + T_new, :] = V_new

        # curr_len is updated once per GPT.forward call, not per layer update

    def get(self, layer_id):
        return self.K[layer_id][:, :, :self.curr_len, :], self.V[layer_id][:, :, :self.curr_len, :]


In [67]:
class SelfAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x, kv_cache=None, layer_id=None):
        B, T, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # reshape for heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)

        if kv_cache is not None and kv_cache.curr_len > 0: # Only retrieve from cache if it's not empty
            K_cached, V_cached = kv_cache.get(layer_id) # K_cached: (B, n_heads, curr_len, head_dim)
            K_all = torch.cat([K_cached, k], dim=2) # Concatenate along sequence length dimension (dim=2)
            V_all = torch.cat([V_cached, v], dim=2)
        else:
            K_all, V_all = k, v # K_all, V_all are just the new k, v

        # attention computation
        attn = (q @ K_all.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = attn @ V_all

        # merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out), k, v # k, v here are the new tokens' k, v (B, n_heads, T, head_head)


In [68]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or dim * 4
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

In [69]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.attn = SelfAttention(dim, n_heads)
        self.mlp = MLP(dim)

    def forward(self, x, kv_cache=None, layer_id=None):
        # Attention block
        attn_out, K_new, V_new = self.attn(self.ln1(x), kv_cache, layer_id)
        x = x + attn_out
        # MLP block
        x = x + self.mlp(self.ln2(x))
        return x, K_new, V_new

In [70]:
class GPT(nn.Module):
    def __init__(self, vocab_size, dim, n_heads, n_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)
        self.max_seq_len = max_seq_len

    def forward(self, input_ids, kv_cache=None):
        B, T = input_ids.shape

        if kv_cache is not None and kv_cache.curr_len > 0:
            # During generation, input_ids is typically (B, 1)
            # The position for this token is the current length of the cache
            pos_start_idx = kv_cache.curr_len
        else:
            # During prefill, input_ids is the full prompt (B, T)
            # The positions are 0 to T-1
            pos_start_idx = 0

        # Create positional embeddings for the current input_ids
        # These positions correspond to the *absolute* positions in the sequence
        positions = torch.arange(pos_start_idx, pos_start_idx + T, device=input_ids.device).unsqueeze(0) # (1, T)
        positions = positions.repeat(B, 1) # Make it (B, T) to match input_ids batch

        x = self.token_emb(input_ids) + self.pos_emb(positions)

        # List to store new K,V for all layers in this forward pass
        new_ks_vs = []

        for layer_id, block in enumerate(self.blocks):
            x, K_new, V_new = block(x, kv_cache, layer_id) # K_new, V_new are (B, n_heads, T, dim_head)
            new_ks_vs.append((K_new, V_new))

        # Update kv_cache after all blocks have processed the current input
        if kv_cache is not None:
            for layer_id, (K_new, V_new) in enumerate(new_ks_vs):
                kv_cache.update(layer_id, K_new, V_new)
            kv_cache.curr_len += T # Update cache length based on new tokens processed

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


In [71]:
def sample_top_k(logits, k=50, temperature=1.0):
    logits = logits / temperature
    top_k_vals, top_k_idx = torch.topk(logits, k)
    probs = torch.softmax(top_k_vals, dim=-1)
    idx = torch.multinomial(probs, 1)
    return top_k_idx.gather(-1, idx)

In [72]:
def generate(model, input_ids, kv_cache, max_new_tokens):
    generated = input_ids.clone()
    for step in range(max_new_tokens):
        # last token only
        x = generated[:, -1:]
        logits = model(x, kv_cache)
        next_token = sample_top_k(logits[:, -1, :])
        generated = torch.cat([generated, next_token.unsqueeze(-1)], dim=1)
    return generated

##Minimal Test Case

In [73]:
import torch
import torch.nn as nn

# Tiny model for testing
vocab_size = 100
dim = 32
n_heads = 4
n_layers = 2
max_seq_len = 16
device = "cuda" if torch.cuda.is_available() else "cpu"

# Assuming batch_size is 1 for this test case based on input_ids later
batch_size = 1

# KV Cache
kv_cache = KVCacheManager(n_layers, n_heads, max_seq_len, dim // n_heads, device, batch_size)

# Model
model = GPT(vocab_size, dim, n_heads, n_layers, max_seq_len).to(device)


In [77]:
# Batch size 1, prompt length 4
input_ids = torch.tensor([[10, 25, 30, 7]], device=device)

In [78]:
# Prefill: compute over full prompt
logits = model(input_ids, kv_cache=kv_cache)
print("Prefill logits shape:", logits.shape)  # Expected: [1, 4, 100]

Prefill logits shape: torch.Size([1, 4, 100])


In [79]:
# Start with last token from prompt
generated = input_ids.clone()
for step in range(3):  # generate 3 new tokens
    x = generated[:, -1:]
    logits = model(x, kv_cache=kv_cache)
    # simple argmax sampling
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated = torch.cat([generated, next_token], dim=1)

print("Generated sequence:", generated)

Generated sequence: tensor([[10, 25, 30,  7, 75, 93, 85]])
